# EDA Bitcoin Alpha

## Pergunta de pesquisa
Como os usuários do Bitcoin Alpha expressam confiança e desconfiança, e de que maneira a presença de avaliações negativas está associada ao volume de avaliações recebidas?

Cada linha é uma avaliação direcionada entre usuários. O sinal da nota representa confiança ou desconfiança.

Fonte: [SNAP — Bitcoin Alpha](https://snap.stanford.edu/data/soc-sign-bitcoin-alpha.html). Base utilizada: CSV da versão 4 do projeto.

## Setup de bibliotecas e dados
Execute na raiz do projeto ou mantenha o CSV ao lado do notebook. Dependências: pandas, numpy, matplotlib, seaborn e scipy.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
POSITIVA, NEGATIVA = "#3A9D75", "#D96C60"
cores = {"Positiva": POSITIVA, "Negativa": NEGATIVA}
candidatos = [Path("data/dataset_parte2/soc-sign-bitcoinalpha.csv"), Path("soc-sign-bitcoinalpha.csv"), Path("soc-sign-bitcoinalpha.csv.gz")]
caminho = next((p for p in candidatos if p.exists()), None)
if caminho is None:
    raise FileNotFoundError("Coloque o CSV do Bitcoin Alpha ao lado do notebook.")
df = pd.read_csv(caminho, header=None, names=["origem", "destino", "nota", "timestamp"])
df.head()

## 1. Princípio estrutural
**Pergunta:** qual é a dimensão da rede e o significado das variáveis? IDs são identificadores, não medidas quantitativas.

In [ ]:
print("Dimensão:", df.shape)
df.info()
print("Duplicadas completas:", df.duplicated().sum())
print("Pares origem-destino repetidos:", df.duplicated(["origem", "destino"]).sum())
usuarios = pd.Index(sorted(set(df.origem) | set(df.destino)))
print("Usuários:", len(usuarios))
print("Notas inválidas:", (~df.nota.between(-10,10) | df.nota.eq(0)).sum())

**Discussão:** são 24.186 registros e 3.783 usuários distintos. Origem e destino representam os extremos de um arco; nota é seu peso e timestamp registra o instante da avaliação. Não há duplicatas completas.

## 2. Dados faltantes
**Pergunta:** existem valores ausentes que exigem imputação?

In [ ]:
df.isna().sum()

Não há valores ausentes nas quatro colunas. Não se aplica imputação nem classificação de mecanismo MCAR/MAR/MNAR. A ausência de um arco não equivale a um dado faltante em uma linha.

In [ ]:
df["data"] = pd.to_datetime(df.timestamp, unit="s", utc=True)
df["ano"] = df.data.dt.year
df["negativa"] = df.nota.lt(0)
df["tipo"] = np.where(df.negativa, "Negativa", "Positiva")
print(df.data.min(), df.data.max())
por_usuario = pd.DataFrame(index=usuarios)
por_usuario["enviadas"] = df.groupby("origem").size().reindex(usuarios, fill_value=0)
por_usuario["recebidas"] = df.groupby("destino").size().reindex(usuarios, fill_value=0)
por_usuario["negativas_recebidas"] = df.groupby("destino").negativa.sum().reindex(usuarios, fill_value=0)
por_usuario["saldo_+/-"] = df.groupby("destino").nota.sum().reindex(usuarios, fill_value=0)
por_usuario.head()

## 3. Princípio distribucional
**Pergunta:** como as notas e a atividade dos usuários se distribuem?

In [ ]:
def descritiva(dados):
    tabela = dados.describe().T
    tabela["assimetria"] = dados.skew()
    tabela["curtose"] = dados.kurt()
    return tabela
antes = descritiva(df[["nota"]])
display(antes.round(3), descritiva(por_usuario).round(3))

In [ ]:
contagem = df.nota.value_counts().sort_index()
contagem.plot.bar(color=[NEGATIVA if x < 0 else POSITIVA for x in contagem.index], figsize=(10,4))
plt.title("Distribuição das notas"); plt.xlabel("Nota"); plt.ylabel("Avaliações"); plt.show()

In [ ]:
atividade = por_usuario.loc[por_usuario.enviadas.gt(0), "enviadas"]
faixas = pd.cut(atividade, [0,10,25,50,100,np.inf], labels=["1–10","11–25","26–50","51–100",">100"])
faixas.value_counts(sort=False).plot.bar(color=POSITIVA)
plt.title("Avaliações realizadas por usuário ativo"); plt.ylabel("Usuários"); plt.xticks(rotation=0); plt.show()

In [ ]:
sns.boxplot(data=df, x="tipo", y="nota", order=["Negativa","Positiva"], hue="tipo", palette=cores, dodge=False)
plt.title("Distribuição das notas por sinal"); plt.show()
display(df.groupby("tipo").nota.describe().round(2))

**Discussão:** as notas positivas predominam e se concentram em valores baixos, enquanto as negativas incluem muitas notas -10. A maioria dos usuários que avaliam realiza poucas avaliações. A nota é discreta e limitada, não uma variável normalmente distribuída.

In [ ]:
amostra = df.nota.sample(min(5000,len(df)), random_state=42)
w,p = stats.shapiro(amostra)
print(f"Shapiro-Wilk (amostra): W={w:.4f}, p={p:.3g}")
stats.probplot(amostra, dist="norm", plot=plt); plt.title("Q-Q plot das notas"); plt.show()
sns.ecdfplot(data=df, x="nota", hue="tipo", palette=cores); plt.title("Distribuição acumulada por sinal"); plt.show()

O teste de normalidade é apenas diagnóstico: as avaliações podem ser dependentes por compartilharem usuários. A distribuição discreta e limitada já desaconselha pressupor normalidade.

## 4. Tratamento de dados ausentes e outliers
**Pergunta:** valores extremos são erros ou características reais da rede?

In [ ]:
q1, q3 = df.nota.quantile([.25,.75])
iqr = q3-q1
lim_inf, lim_sup = q1-1.5*iqr, q3+1.5*iqr
outliers = df.loc[~df.nota.between(lim_inf,lim_sup)]
print("Limites IQR:", lim_inf, lim_sup)
print("Notas sinalizadas:", len(outliers))
display(outliers.nota.value_counts().sort_index())
sns.boxplot(x=por_usuario.recebidas); plt.title("Avaliações recebidas por usuário"); plt.show()

Notas extremas dentro da escala válida e usuários muito ativos são informações relevantes, não erros comprovados. Não removemos nem imputamos registros. Uma escala logarítmica pode ajudar a visualizar graus, sem alterar o grafo original.

In [ ]:
plt.hist(np.log1p(por_usuario.recebidas), bins=30, color=POSITIVA)
plt.xlabel("log(1 + avaliações recebidas)"); plt.title("Conectividade em escala logarítmica"); plt.show()
df_tratado = df.copy()

## 5. Estatística descritiva — antes e depois

In [ ]:
depois = descritiva(df_tratado[["nota"]])
display(pd.concat({"Antes": antes, "Depois": depois, "Diferença": depois-antes}).round(3))
assert df_tratado.equals(df)

A diferença é zero porque não havia justificativa para modificar as observações. As variáveis derivadas preservam os registros originais.

## 6. Princípio relacional
### Frequências e desbalanceamento
**Pergunta:** qual sinal predomina?

In [ ]:
freq = df.tipo.value_counts()
display(pd.DataFrame({"Quantidade":freq,"Percentual":100*freq/len(df)}).round(2))
print("Entropia normalizada:", stats.entropy(freq, base=2))

A confiança predomina: 22.650 avaliações positivas e 1.536 negativas. O desbalanceamento é uma característica observada da rede, não motivo para reamostrar a EDA.

### Evolução temporal
**Pergunta:** como o volume de avaliações muda ao longo do tempo?

In [ ]:
df.groupby("ano").size().plot.bar(color=POSITIVA)
plt.title("Avaliações por ano"); plt.ylabel("Quantidade"); plt.xticks(rotation=0); plt.show()
df.set_index("data").resample("MS").size().plot(figsize=(11,4))
plt.title("Evolução mensal das avaliações"); plt.ylabel("Quantidade"); plt.show()

O volume é maior em 2011 e diminui nos anos seguintes. Os anos extremos têm cobertura parcial. Volume absoluto e proporção de negativas respondem a perguntas diferentes.

### Análise bivariada e multivariada
**Pergunta:** usuários mais avaliados recebem mais negativas?

In [ ]:
recebedores = por_usuario.loc[por_usuario.recebidas.gt(0)].copy()
sns.scatterplot(data=recebedores, x="recebidas", y="negativas_recebidas", alpha=.45)
plt.title("Total recebido x avaliações negativas recebidas"); plt.show()
variaveis = ["enviadas", "recebidas", "negativas_recebidas", "saldo_recebido"]
display(recebedores[variaveis].corr(method="pearson").round(3))
sns.heatmap(recebedores[variaveis].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1,vmax=1)
plt.title("Correlação de Spearman"); plt.show()

A dispersão permite identificar usuários muito avaliados com poucas negativas e outros com muitas negativas. O número de negativas é parte do total recebido; a correlação não é evidência independente de causalidade. IDs e timestamps brutos não entram na matriz. Não há atributos demográficos para avaliar fairness entre grupos protegidos.

## 7. Visualizações explanatórias



A identidade visual reutiliza `estilo_graficos.py`: azul para avaliações positivas ou ausência de negativas, vermelho para presença de negativas, fundo discreto e títulos orientados às descobertas. Similaridade agrupa categorias; proximidade associa rótulos aos dados; continuidade organiza intensidades e faixas; figura e fundo destacam as evidências. Valores escritos, sinais e legendas reduzem a dependência da cor.



In [ ]:
from src.avd.visualizacoes_explanatorias import gerar_visualizacoes, resumir_avaliacoes_por_faixa
from IPython.display import Image, display
pasta = Path("out/parte2/avd/visualizacoes_explanatorias")
gerar_visualizacoes(saida=pasta)
for nome in ["intensidade_confianca_desconfianca.png", "conectividade_desconfianca.png"]:
    display(Image(filename=str(pasta / nome)))
usuarios, faixas_volume = resumir_avaliacoes_por_faixa(df)
display(faixas_volume)

## Visualização 1 — Intensidade das notas

Barras agrupadas por intensidade absoluta, com percentuais calculados separadamente por sinal. Entre as 22.650 positivas, 60,75% são +1; entre as 1.536 negativas, 52,86% são −10 e 27,93% são −1. A confiança concentra-se em baixa intensidade; a desconfiança tem concentração no extremo máximo, mas também no mínimo.

## Visualização 2 — Presença de negativas por volume recebido

Barras empilhadas de 100%, com usuários agrupados por total recebido. A proporção com pelo menos uma negativa é 11,85% (1–10), 48,04% (11–25), 53,64% (26–50), 48,98% (51–100) e 59,09% (>100). Os grupos maiores têm poucos usuários. Entre os 3.754 usuários avaliados, os 630 com negativas recebem em média 15,63 avaliações, contra 4,59 dos 3.124 sem negativas. Maior exposição também aumenta a oportunidade de receber uma negativa.

## Big Idea

No Bitcoin Alpha, a confiança concentra-se em notas de baixa intensidade, enquanto a desconfiança frequentemente aparece no extremo da escala e é mais comum no histórico dos usuários mais avaliados.

## Storytelling — 3-Minute Story

Será que confiança e desconfiança são expressas da mesma maneira? A nota +1 predomina entre as positivas, enquanto −10 predomina entre as negativas. Ao agrupar usuários pelo volume recebido, observamos que a presença de negativas passa de cerca de 12% na faixa de 1 a 10 avaliações para 48% na faixa de 11 a 25. Nas demais faixas, permanece acima da primeira, sem crescimento contínuo. Assim, a intensidade das notas e a presença de negativas revelam padrões distintos de avaliação.

## Conclusão

Os resultados são descritivos, não causais. Mais avaliações aumentam as oportunidades de receber uma negativa; receber uma negativa não comprova comportamento inadequado. Os experimentos de caminhos e ciclos permanecem como análises complementares de grafos.
